# LLM vs Human Discrimination Analysis
**Research Question:** Can LLMs reliably distinguish real from synthetic tabular data,
and how does their performance compare to humans across two information conditions?

- **C1**: Table only
- **C2**: Table + distributional metadata

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Aesthetics
plt.rcParams.update({
    'figure.facecolor': '#0d0f14',
    'axes.facecolor': '#13161e',
    'axes.edgecolor': '#2a2d38',
    'axes.labelcolor': '#e8e6e0',
    'text.color': '#e8e6e0',
    'xtick.color': '#9ca3af',
    'ytick.color': '#9ca3af',
    'grid.color': '#2a2d38',
    'font.family': 'monospace',
})
ACCENT = '#c8a96e'
PALETTE = ['#60a5fa', '#c084fc', '#4ade80', '#f87171', '#fbbf24']

RESULTS_DIR = Path('../results')
print('Results files:', list(RESULTS_DIR.glob('*.jsonl')))

## 1. Load Results

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

# Load LLM results (all providers)
llm_files = list(RESULTS_DIR.glob('llm_results_*.jsonl'))
llm_dfs = [load_jsonl(f) for f in llm_files]
llm_df = pd.concat(llm_dfs, ignore_index=True) if llm_dfs else pd.DataFrame()

# Load human results
human_path = RESULTS_DIR / 'human_results.jsonl'
human_df = load_jsonl(human_path) if human_path.exists() else pd.DataFrame()

print(f'LLM results: {len(llm_df)} trials')
print(f'Human results: {len(human_df)} trials')

if not llm_df.empty:
    display(llm_df.head())
if not human_df.empty:
    display(human_df.head())

## 2. Overall Accuracy

In [ ]:
def accuracy_summary(df, label=''):
    if df.empty:
        return pd.DataFrame()
    g = df.groupby(['condition', 'true_label'])['correct'].agg(['mean', 'sum', 'count'])
    g.columns = ['Accuracy', 'Correct', 'Total']
    g['Source'] = label
    return g.reset_index()

llm_acc = accuracy_summary(llm_df, 'LLM')
human_acc = accuracy_summary(human_df, 'Human')
combined = pd.concat([llm_acc, human_acc], ignore_index=True)
display(combined)

## 3. RQ1 — Effect of Metadata (C1 vs C2)

In [ ]:
def plot_condition_comparison(df, title, color):
    if df.empty or 'condition' not in df.columns:
        print(f'No data for: {title}')
        return
    grouped = df.groupby('condition')['correct'].mean().reset_index()
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(grouped['condition'], grouped['correct'], color=color, alpha=0.85, width=0.4)
    ax.set_ylim(0, 1)
    ax.axhline(0.5, color='#f87171', linestyle='--', linewidth=1, label='Chance (0.5)')
    ax.set_ylabel('Accuracy')
    ax.set_title(title, color=ACCENT, fontsize=13, fontweight='bold')
    ax.legend()
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{bar.get_height():.2f}', ha='center', fontsize=10)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'condition_comparison_{title.lower().replace(" ","_")}.png', dpi=150)
    plt.show()

plot_condition_comparison(llm_df, 'LLM Condition Comparison', PALETTE[0])
plot_condition_comparison(human_df, 'Human Condition Comparison', PALETTE[1])

## 4. RQ2 — LLM vs Human Head-to-Head

In [ ]:
def head_to_head(llm_df, human_df):
    rows = []
    for cond in ['C1', 'C2']:
        for label in ['REAL', 'SYNTHETIC']:
            for source, df in [('LLM', llm_df), ('Human', human_df)]:
                if df.empty:
                    continue
                sub = df[(df['condition'] == cond) & (df['true_label'] == label)]
                if len(sub) == 0:
                    continue
                rows.append({
                    'Condition': cond,
                    'True Label': label,
                    'Source': source,
                    'Accuracy': sub['correct'].mean(),
                    'N': len(sub),
                })
    return pd.DataFrame(rows)

h2h = head_to_head(llm_df, human_df)
display(h2h)

if not h2h.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for i, cond in enumerate(['C1', 'C2']):
        sub = h2h[h2h['Condition'] == cond]
        ax = axes[i]
        x = np.arange(len(sub['True Label'].unique()))
        sources = sub['Source'].unique()
        width = 0.35
        for j, src in enumerate(sources):
            src_data = sub[sub['Source'] == src].set_index('True Label')['Accuracy']
            ax.bar(x + j*width, src_data.values, width, label=src, color=PALETTE[j], alpha=0.85)
        ax.set_xticks(x + width/2)
        ax.set_xticklabels(sub['True Label'].unique())
        ax.set_ylim(0, 1)
        ax.axhline(0.5, color='#f87171', linestyle='--', linewidth=1)
        ax.set_title(f'{cond} — LLM vs Human', color=ACCENT, fontweight='bold')
        ax.set_ylabel('Accuracy')
        ax.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'llm_vs_human_head2head.png', dpi=150)
    plt.show()

## 5. Confidence Calibration (LLM)

In [ ]:
if not llm_df.empty and 'confidence' in llm_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for i, cond in enumerate(['C1', 'C2']):
        sub = llm_df[llm_df['condition'] == cond].copy()
        sub = sub[sub['confidence'] >= 0]
        if sub.empty:
            continue
        bins = pd.cut(sub['confidence'], bins=[0, 20, 40, 60, 80, 100])
        cal = sub.groupby(bins)['correct'].mean()
        ax = axes[i]
        cal.plot(kind='bar', ax=ax, color=PALETTE[0], alpha=0.85)
        ax.set_title(f'{cond} — Confidence Calibration', color=ACCENT, fontweight='bold')
        ax.set_xlabel('Confidence bucket')
        ax.set_ylabel('Accuracy')
        ax.set_ylim(0, 1)
        ax.axhline(0.5, color='#f87171', linestyle='--', linewidth=1)
        ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'llm_calibration.png', dpi=150)
    plt.show()

## 6. Statistical Tests

In [ ]:
def binomial_test(correct, total, p0=0.5):
    """One-sided binomial test: H1: accuracy > chance."""
    result = stats.binomtest(correct, total, p=p0, alternative='greater')
    return result.pvalue

for source, df in [('LLM', llm_df), ('Human', human_df)]:
    if df.empty:
        continue
    print(f'\n=== {source} ===')
    for cond in df['condition'].unique():
        sub = df[df['condition'] == cond]
        correct = sub['correct'].sum()
        total = len(sub)
        acc = correct / total
        p_val = binomial_test(correct, total)
        sig = '**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns')
        print(f'  {cond}: acc={acc:.3f} ({correct}/{total}), p={p_val:.4f} {sig}')

## 7. LLM Reasoning Quality — Red Flags Analysis

In [ ]:
if not llm_df.empty and 'red_flags' in llm_df.columns:
    # Flatten red flags
    all_flags = []
    for _, row in llm_df.iterrows():
        flags = row.get('red_flags', [])
        if isinstance(flags, list):
            for flag in flags:
                all_flags.append({
                    'flag': flag,
                    'correct': row['correct'],
                    'true_label': row['true_label'],
                    'condition': row['condition'],
                })
    flags_df = pd.DataFrame(all_flags)
    print(f'Total red flags cited: {len(flags_df)}')
    print(f'Avg flags per trial: {len(flags_df)/len(llm_df):.2f}')
    
    # Show sample flags from correct vs incorrect trials
    print('\n--- Sample red flags (correct trials) ---')
    correct_flags = flags_df[flags_df['correct'] == True]['flag'].sample(min(5, len(flags_df[flags_df['correct']==True]))).tolist()
    for f in correct_flags:
        print(f'  • {f}')
    
    print('\n--- Sample red flags (incorrect trials) ---')
    wrong_flags = flags_df[flags_df['correct'] == False]['flag'].sample(min(5, len(flags_df[flags_df['correct']==False]))).tolist()
    for f in wrong_flags:
        print(f'  • {f}')

## 8. Export Summary

In [ ]:
summary_rows = []
for source, df in [('LLM', llm_df), ('Human', human_df)]:
    if df.empty:
        continue
    for cond in df['condition'].unique():
        for label in df['true_label'].unique():
            sub = df[(df['condition'] == cond) & (df['true_label'] == label)]
            if len(sub) == 0:
                continue
            correct = sub['correct'].sum()
            total = len(sub)
            p_val = binomial_test(correct, total)
            summary_rows.append({
                'Source': source,
                'Condition': cond,
                'True Label': label,
                'N': total,
                'Correct': int(correct),
                'Accuracy': round(correct/total, 4),
                'p_value': round(p_val, 4),
                'sig': '**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'),
            })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
summary_df.to_csv(RESULTS_DIR / 'final_summary.csv', index=False)
print('Saved to results/final_summary.csv')